# Launch & Record — 팀원용 단일 노트북

이 노트북 한 개로 **카메라 + 모터 + bag 레코더**를 켜고 두 카메라 화면을 보면서 녹화한다.

**전제 — 한 번만 빌드** (터미널에서):
```bash
cd /home/ircv16/team/final_project/ros2_ws
colcon build --symlink-install
source install/setup.bash
```
그 후 이 노트북을 같은 셸 환경에서 `jupyter notebook`으로 실행 (`ros2` 명령이 PATH에 있어야 함).

**실행 순서:**
1. Config + ROS init
2. **Launch** — `ros2 launch rover_recorder record.launch.py` 실행 (카메라+모터+bag 노드)
3. **Preview** — 두 카메라 라이브 화면
4. **Teleop 안내** — 별도 SSH 터미널에서 `ros2 run rover_teleop teleop_node` 실행
5. **Stop** — 끝낼 때

**왜 teleop만 따로 터미널?** cbreak 키 입력은 자기 TTY를 점유해야 동작. 노트북 subprocess로 띄우면 키가 안 먹힘. 카메라/모터/bag은 노트북에서 띄워도 무관.

## 1. Config + ROS init

In [5]:
SESSION_NAME = 'session01'
OUT_ROOT     = '/home/ircv16/team/final_project/rover_data'
FPS          = 15
DRY_RUN      = False    # True면 모터 안 움직임 (UART 안 씀)
MONITOR      = True     # True면 launch가 브라우저 MJPEG 모니터(rover_monitor)도 띄움

# 이 워크스페이스 (rover_recorder 등 우리 패키지가 사는 곳)
WS = '/home/ircv16/team/final_project/ros2_ws'
ROS_DISTRO_SETUP = '/opt/ros/humble/setup.bash'

import os, sys, time, signal, subprocess, threading, shutil
from pathlib import Path


def _source_into_environ(*setup_scripts):
    """bash로 setup.bash들을 source한 뒤 결과 환경변수를 이 파이썬
    프로세스(os.environ)에 반영. VSCode/jupyter를 어디서 띄웠든
    커널이 우리 워크스페이스 패키지를 찾게 해준다."""
    srcs = ' && '.join(f'source "{s}"' for s in setup_scripts if os.path.exists(s))
    if not srcs:
        raise RuntimeError(f'source 대상 setup.bash 없음: {setup_scripts}')
    out = subprocess.run(
        ['bash', '-c', f'{srcs} && env -0'],
        capture_output=True,
    )
    if out.returncode != 0:
        raise RuntimeError(f'source 실패: {out.stderr.decode(errors="replace")}')
    for entry in out.stdout.split(b'\x00'):
        if not entry:
            continue
        k, _, v = entry.partition(b'=')
        os.environ[k.decode()] = v.decode()


_source_into_environ(ROS_DISTRO_SETUP, f'{WS}/install/setup.bash')

# ros2 자체
if shutil.which('ros2') is None:
    raise RuntimeError(
        'ros2 not on PATH. /opt/ros/humble/setup.bash 가 source 안 됐습니다. '
        f'ROS_DISTRO_SETUP={ROS_DISTRO_SETUP} 경로 확인하세요.'
    )

# 우리 워크스페이스가 실제로 잡혔는지 (rover_recorder 보이는지) 검사
_chk = subprocess.run(['ros2', 'pkg', 'prefix', 'rover_recorder'],
                      capture_output=True, text=True)
if _chk.returncode != 0:
    raise RuntimeError(
        "rover_recorder 패키지를 못 찾음. 워크스페이스 빌드/소스 문제.\n"
        f"  - WS={WS}\n"
        f"  - {WS}/install/setup.bash 존재? {os.path.exists(WS + '/install/setup.bash')}\n"
        "빌드부터: cd $WS && colcon build --symlink-install\n"
        f"stderr: {_chk.stderr.strip()}"
    )

import rclpy
if not rclpy.ok():
    rclpy.init()
print('ros2:', shutil.which('ros2'))
print('rover_recorder prefix:', _chk.stdout.strip())
print('rclpy ok')

ros2: /opt/ros/humble/bin/ros2
rover_recorder prefix: /home/ircv16/team/final_project/ros2_ws/install/rover_recorder
rclpy ok


In [6]:
import subprocess
print(subprocess.run(['ros2','pkg','prefix','rover_recorder'],
                     capture_output=True, text=True).stderr or 'OK: 패키지 보임')
import os
print('AMENT:', os.environ.get('AMENT_PREFIX_PATH','(없음)'))


OK: 패키지 보임
AMENT: /home/ircv16/team/final_project/ros2_ws/install/rover_teleop:/home/ircv16/team/final_project/ros2_ws/install/rover_recorder:/home/ircv16/team/final_project/ros2_ws/install/rover_camera:/home/ircv16/team/final_project/ros2_ws/install/rover_calib:/opt/ros/humble


## 2. Launch — 카메라 + 모터 + bag 레코더

`ros2 launch rover_recorder record.launch.py`를 subprocess로 띄움. 로그는 아래 출력 영역에 흘러나옴.

**이미 떠 있는 ros2 프로세스가 있으면 충돌**할 수 있으니, 처음 실행 시 다른 노드는 다 죽이고 시작.

In [7]:
import ipywidgets as widgets
from IPython.display import display

launch_proc = {'p': None}
_launch_lock = threading.Lock()   # 동시 클릭으로 launch 2개 뜨는 것 방지

log_output = widgets.Output(layout=widgets.Layout(
    border='1px solid gray', height='250px', overflow='auto'))
display(log_output)

def _pump(p):
    for line in iter(p.stdout.readline, b''):
        with log_output:
            print(line.decode(errors='replace').rstrip())

def start_launch():
    # 락 안에서 검사+기록을 한 번에 → 거의 동시에 두 번 눌려도 한 번만 뜬다.
    with _launch_lock:
        p = launch_proc['p']
        if p is not None and p.poll() is None:
            with log_output:
                print('already running (무시)')
            return
        cmd = [
            'ros2', 'launch', 'rover_recorder', 'record.launch.py',
            f'session_name:={SESSION_NAME}',
            f'out_root:={OUT_ROOT}',
            f'fps:={FPS}',
            f'dry_run:={str(DRY_RUN).lower()}',
            f'monitor:={str(MONITOR).lower()}',
        ]
        with log_output:
            print('LAUNCH:', ' '.join(cmd))
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,
        )
        launch_proc['p'] = proc
    threading.Thread(target=_pump, args=(proc,), daemon=True).start()

def stop_launch():
    with _launch_lock:
        p = launch_proc['p']
        if p is None or p.poll() is not None:
            with log_output:
                print('not running')
            launch_proc['p'] = None
            return
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGINT)
            p.wait(timeout=8)
        except Exception:
            try: p.kill()
            except Exception: pass
        launch_proc['p'] = None
        with log_output:
            print('launch stopped')

# 디바운스: 클릭 직후 버튼을 잠깐 비활성화해서 더블클릭/중복 이벤트 차단.
def _guarded(fn, btn):
    def handler(_):
        btn.disabled = True
        try:
            fn()
        finally:
            def _reenable():
                time.sleep(1.5); btn.disabled = False
            threading.Thread(target=_reenable, daemon=True).start()
    return handler

btn_start = widgets.Button(description='▶ START launch', button_style='success')
btn_stop  = widgets.Button(description='■ STOP launch',  button_style='danger')
btn_start.on_click(_guarded(start_launch, btn_start))
btn_stop.on_click(_guarded(stop_launch, btn_stop))
display(widgets.HBox([btn_start, btn_stop]))

Output(layout=Layout(border_bottom='1px solid gray', border_left='1px solid gray', border_right='1px solid gra…

## 3. Preview — 브라우저 모니터 (localhost MJPEG)

미리보기는 이제 노트북 위젯이 아니라 **별도 웹페이지**에서 본다. `record.launch.py`가
`rover_monitor` 노드를 같이 띄우며, 이 노드는 카메라가 보내는 JPEG 바이트를 **재인코딩 없이 그대로**
MJPEG로 흘린다 → 디코딩은 브라우저가 하므로 노트북 위젯보다 표시 지연이 낮다.

**열기:** 브라우저에서 아래 주소.
- 젯슨 화면에서 직접: `http://localhost:8080/`
- 같은 네트워크의 노트북에서: `http://<젯슨-IP>:8080/`  (기본 `monitor_host:=0.0.0.0`)

`http://localhost:8080/` 한 페이지에 BEV/FRONT가 나란히 뜬다. 끊기면 launch 로그의
`frame age` 값이 올라간다 (정상은 0.0x초).

> **주행 타이밍과 무관:** 모니터는 제어 경로(teleop→/cmd_vel→motor_bridge→UART)와 완전히 별개 노드라,
> 화면이 얼마나 빠르든/느리든 모터 명령 지연에 영향이 없다.

아래 셀은 모니터 주소를 클릭 링크로 띄워주는 용도(선택).

In [8]:
import socket
from IPython.display import display, HTML

MONITOR_PORT = 8080

def _lan_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(('8.8.8.8', 80)); return s.getsockname()[0]
    except Exception:
        return 'localhost'
    finally:
        s.close()

ip = _lan_ip()
display(HTML(
    f'<p>모니터 페이지 (launch가 떠 있어야 함):</p>'
    f'<ul>'
    f'<li>젯슨 로컬: <a href="http://localhost:{MONITOR_PORT}/" target="_blank">'
    f'http://localhost:{MONITOR_PORT}/</a></li>'
    f'<li>같은 네트워크: <a href="http://{ip}:{MONITOR_PORT}/" target="_blank">'
    f'http://{ip}:{MONITOR_PORT}/</a></li>'
    f'</ul>'
))

## 4. Teleop 안내 — 별도 SSH 터미널에서 실행

이 노트북과 **다른 SSH 터미널**을 열고:
```bash
cd /home/ircv16/team/final_project/ros2_ws
source install/setup.bash
ros2 run rover_teleop teleop_node
```

**키 매핑** (teleop 터미널에서):
- `a` / `d` : turn_level −1 / +1 (−2..+2, 두 번이면 최대 회전)
- `space` : 정지
- `g` : drive on/off (모터 송신 토글)
- `r` : 녹화 on/off (bag 시작/종료)
- `q` / `ESC` : 종료

녹화 toggle 상태는 아래 indicator로 확인 가능.

In [5]:
import threading, time
import rclpy
from rclpy.node import Node
from std_msgs.msg import Bool
import ipywidgets as widgets
from IPython.display import display

rec_state = {'on': False}
rec_stop  = {'stop': False}

class RecIndicator(Node):
    def __init__(self):
        super().__init__('rover_rec_indicator_nb')
        self.create_subscription(Bool, '/record_enable',
                                 lambda m: rec_state.update(on=bool(m.data)), 10)

rec_node = RecIndicator()
def _spin_rec():
    while not rec_stop['stop'] and rclpy.ok():
        rclpy.spin_once(rec_node, timeout_sec=0.1)
threading.Thread(target=_spin_rec, daemon=True).start()

rec_label = widgets.HTML()
display(rec_label)
def _rec_loop():
    while not rec_stop['stop']:
        color = 'red' if rec_state['on'] else 'gray'
        word  = 'RECORDING' if rec_state['on'] else 'idle'
        rec_label.value = (f"<span style='color:{color};font-weight:bold;font-size:18px'>"
                           f"● {word}</span>")
        time.sleep(0.2)
threading.Thread(target=_rec_loop, daemon=True).start()
print('rec indicator running')

HTML(value='')

rec indicator running


## 5. Stop — 끝낼 때

**순서 중요**: teleop 터미널에서 먼저 `q`로 종료해서 모터 정지 시킨 다음, 아래 셀로 launch 종료.

In [5]:
rec_stop['stop'] = True
time.sleep(0.3)
stop_launch()
try: rec_node.destroy_node()
except Exception: pass
if rclpy.ok():
    rclpy.shutdown()
print('done')

NameError: name 'rec_stop' is not defined